In [49]:
from count_variants import count_variants_by_type

In [50]:
my_vcf1="../vcf_files/vfaba_hedin_peamust_SNP_GATK.vcf.gz"
my_vcf2="../vcf_files/PEAMUST_vfaba_F1_SNP_imputed_MAF5.vcf.gz"
my_test_vcf="../vcf_files/test.vcf.gz"

In [51]:
def count_variants_by_type(vcf_file_path, n_threads=1, step_name="step01"):
    """
    Count the number of variants for each type in a VCF file using cyvcf2.
    Args:
        vcf_file_path (str): Path to the VCF file.
        n_threads (int): Number of threads to use for reading the VCF file.
        step_name (str): Name of the step used in the column "step".
    Returns:
        Pandas DataFrame: A DataFrame with the counts of SNPs and InDels.
    """
    vcf_reader = VCF(vcf_file_path, threads=n_threads)
    snp_variant_count = 0
    indel_variant_count = 0
    sv_variant_count = 0

    for variant in vcf_reader:
        # Check if the variant is a SNP
        if len(variant.REF) == 1 and len(variant.ALT[0]) == 1:
            snp_variant_count += 1
        # Check if the variant is an indel
        elif len(variant.REF) != len(variant.ALT[0]):
            indel_variant_count += 1
        else:   
            pass # SVs are not counted in this function

    vcf_reader.close()

    # Create a DataFrame to hold the counts
    variant_counts = {
        "step": [step_name],
        "snp_variant_count": [snp_variant_count],
        "indel_variant_count": [indel_variant_count],
        "sv_variant_count": [sv_variant_count]
    }
    variant_counts_df = pd.DataFrame.from_dict(variant_counts, orient='columns')
    return variant_counts_df


In [52]:
df = count_variants_by_type(my_test_vcf, n_threads=1)

In [53]:
df.head()

,step,snp_variant_count,indel_variant_count,sv_variant_count
0,step01,5960,0,0


In [72]:
count_csv_files = ["../scratch/" + f for f in ["counts/vfaba.step0.csv","counts/vfaba.step1.csv","counts/vfaba.step2.csv"]]

In [73]:
count_csv_files

['../scratch/counts/vfaba.step0.csv',
 '../scratch/counts/vfaba.step1.csv',
 '../scratch/counts/vfaba.step2.csv']

In [ ]:
counts_df = []
for f in count_csv_files:
    df = pd.read_csv(f, index_col=0).head()
    counts_df.append(df)
counts_df = pd.concat(counts_df, axis=0)
counts_df.to_csv(output, index=False)




,snp_variant_count,indel_variant_count,sv_variant_count
step,,,
step0: raw file,5960,0,0
step1: biallelic SNPs,5114,0,0
step2: first filters on SNPs,1727,0,0


In [ ]:
import pandas as pd
merged_df = pd.concat(all_counts, ignore_index=True)
merged_df.to_csv(output, index=False)